# L6: Full Stack Agent 🎨

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> , notebooks and other files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>

</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

First start by filtering warnings and loading environment variables.

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from helper import load_env

load_env()

To create this full stack agent, you will provide the following tools to your agent.


1. `list_directory`: List directories
2. `read_file`: Read files
3. `write_file`: Write files
4. `search_file_content`: Search file content
5. `replace_in_file`: Replace in file
6. `glob`: Search files and directories

You can test the `search_file_content` tool by importing it below. It will search the files for instances of "sbx" and return paginated results.

In [2]:
from lib.sbx_tools import search_file_content

search_file_content("sbx", limit=4)

{'pagination': {'total': 34, 'offset': 0, 'limit': 4, 'has_more': True},
 'results': [{'file': './L6.ipynb',
   'line': 82,
   'content': '"You can test the `search_file_content` tool by importing it below. It will search the files for instances of \\"sbx\\" and return paginated results."'},
  {'file': './L6.ipynb',
   'line': 92,
   'content': '"from lib.sbx_tools import search_file_content\\n",'},
  {'file': './L6.ipynb',
   'line': 94,
   'content': '"search_file_content(\\"sbx\\", limit=4)"'},
  {'file': './L6.ipynb',
   'line': 145,
   'content': '"sbx = create_sandbox(template=\\"dlai-nextjs-developer\\")\\n",'}],
 'files_searched': 21}

In [4]:
cat lib/sbx_tools.py

"""
Sandbox Tools - File system operations for e2b sandbox
Clean, fast, throws proper exceptions.
"""

import os
import fnmatch
import glob as glob_module
import re
from typing import Dict, List, Any, Optional


class ToolError(Exception):
    """Custom exception for tool failures - gets caught by e2b execution"""

    pass


def _paginate_results(
    results: List[Any], offset: int = 0, limit: Optional[int] = 16
) -> Dict[str, Any]:
    """Slice results and return pagination metadata"""
    total = len(results)
    start = max(0, offset)
    limit = min(limit, 64)
    end = start + limit if limit else total
    page = results[start:end]

    return {
        "pagination": {
            "total": total,
            "offset": start,
            "limit": limit,
            "has_more": end < total,
        },
        "results": page[start:end],
    }


def secure_path(requested_path: str) -> str:
    """Keep paths locked to working_dir or die trying"""
    working_dir = os.getcwd()
    wd

## Creating Web Apps

### System Prompt

For full stack agents, more detailed and complicated prompts are required. Check out the system prompt below.

In [5]:
from lib.prompts import SYSTEM_PROMPT_WEB_DEV
print(SYSTEM_PROMPT_WEB_DEV)

You are a Senior Nextjs programmer. Your purpose is to accomplish tasks by using the set of available tools.

You MUST follow a strict 'Reason then Act' cycle for every turn:

1.  **Reason:** First, think step-by-step about the user's request, your plan, and any previous tool results. Write this reasoning inside a `<scratchpad>` block. This is your private workspace.

2.  **Act:** After you have a clear plan in your thought process, you MUST use one of your available tools to execute the first step of your plan.

If you have completed the task and no more tools are needed, provide a final answer to the user in plain text, without any `<scratchpad>` block or tool calls.

You are currently inside /home/user/ where the nextjs app is, you can only read/edit files there. The app was generated using the following commands

```bash
bunx create-next-app@15.5.0 . --ts --tailwind --no-eslint --import-alias "@/*" --yes
bunx shadcn@2.10.0 init -b neutral -y
bunx shadcn@2.10.0 add --all
```

The pr

In [6]:
cat lib/prompts.py

SYSTEM_PROMPT_COMPRESS_MESSAGES = r"""You are the component that summarizes internal chat history into a given structure.

When the conversation history grows too large, you will be invoked to distill the entire history into a concise, structured XML snapshot. This snapshot is CRITICAL, as it will become the agent's *only* memory of the past. The agent will resume its work based solely on this snapshot. All crucial details, plans, errors, and user directives MUST be preserved.

First, you will think through the entire history in a private <scratchpad>. Review the user's overall goal, the agent's actions, tool outputs, file modifications, and any unresolved questions. Identify every piece of information that is essential for future actions.

After your reasoning is complete, generate the final <state_snapshot> XML object. Be incredibly dense with information. Omit any irrelevant conversational filler.

The structure MUST be as follows:

<state_snapshot>
    <overall_goal>
        <!-- A

Let's get your full stack agent started! You will provide the tools, tool schemas, system prompts, UI, OpenAI client, and create the sandbox. This time you will use a template with Next.js installed.

In [7]:
from lib.coding_agent import (
    coding_agent,
)
from lib.tools_schemas import tools_schemas
from lib.tools import tools
from lib.prompts import SYSTEM_PROMPT_WEB_DEV
from lib.ui import ui
from openai import OpenAI
from lib.utils import create_sandbox

sbx = create_sandbox(template="dlai-nextjs-developer")

client = OpenAI()

messages = []

demo = ui(
    coding_agent,
    messages,
    host=f"https://{sbx.get_host(3000)}",
    client=client,
    sbx=sbx,
    max_steps=100,
    system=SYSTEM_PROMPT_WEB_DEV,
    tools=tools,
    tools_schemas=tools_schemas,
    model="gpt-5-mini",
)
demo.launch(share=True, height=800)

INFO     [sandbox] 🚀 Creating new Sandbox.create(id=iarwgxmy5ge7nkhy2htjm)

INFO     [sandbox] 🔧 Setting it up ...

INFO     [sandbox]       installing packages ...

INFO     [sandbox]       copying tools ...

INFO     [sandbox] 🔧 done!

* Running on local URL:  https://0.0.0.0:7860
* Running on public URL: https://5bb7dbd9bd8a557195.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
cat lib/coding_agent.py

import json
from openai import OpenAI
from typing import Generator, Literal, Optional, Callable
from e2b_code_interpreter import Sandbox
import re
from .logger import logger, log_tool_call
from .prompts import *
from IPython.display import Image, display
import base64
from .tools import execute_tool


TOKEN_LIMIT = 60_000
COMPRESS_THRESHOLD = 0.7
STATE_SNAPSHOT_PATTERN = re.compile(
    r"<state_snapshot>(.*?)</state_snapshot>", re.DOTALL
)


def clean_messages_for_llm(messages: list[dict]) -> list[dict]:
    return [{k: v for k, v in msg.items() if not k.startswith("_")} for msg in messages]


def compress_messages(client: OpenAI, messages: list[dict]) -> list[dict]:
    response = client.responses.create(
        model="gpt-5-nano",
        input=[
            {"role": "developer", "content": SYSTEM_PROMPT_COMPRESS_MESSAGES},
            *messages,
            {
                "role": "user",
                "content": "First, reason in your scratchpad. Then, generate the <state_sna